# Script 13 — Volatility Sensitivity and Regime-Stability Analysis

**Purpose:** evaluate the frozen Script 12 volatility strategy on the existing 2019–2023 outer evaluation predictions. This script does **not** refit models, choose new models, change guardrails, or load the 2024–2025 final holdout.

## Analyses produced
1. Recalculate pooled and yearly QLIKE / variance MSE / volatility RMSE.
2. Check whether the saved Script 12 pooled results reproduce from the final prediction file.
3. Compare 2022 with the other development evaluation years (2022 remains in the primary results).
4. Compare performance on non-fallback origins only as a **secondary diagnostic**, not as a replacement for the primary results.
5. Evaluate normal/high volatility regimes using trailing 20-day return volatility, with each year's threshold estimated only from data available before that validation year.
6. Summarize EWMA fallback rates, selected strategies, and logged model-fit/forecast issues.
7. Save a quality-gate table and figures for reporting.

**Important:** Run Script 12 successfully first. This script expects its saved final predictions and development dataset. It only writes files beginning with `13_` and will not overwrite Script 12 outputs.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

CURRENCIES = ["USD", "GBP", "EUR", "JPY"]
HORIZONS = [1, 3, 7]
EPS = 1e-10
HIGH_VOL_PERCENTILE = 0.75

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents[:5]]
    for root in candidates:
        if (root / "data" / "processed" / "02_development_model_dataset.csv").exists():
            return root.resolve()
    raise FileNotFoundError(
        "Could not find data/processed/02_development_model_dataset.csv. "
        "Open this notebook inside your Research project, or select the correct VS Code folder."
    )

ROOT = find_project_root()
DEV_PATH = ROOT / "data" / "processed" / "02_development_model_dataset.csv"
PRED_DIR = ROOT / "outputs" / "predictions"
TABLE_DIR = ROOT / "outputs" / "tables"
FIG_DIR = ROOT / "outputs" / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PATH = PRED_DIR / "12_volatility_final_predictions_2019_2023.csv"
POOLED_PATH = TABLE_DIR / "12_volatility_pooled_results.csv"
YEARLY_PATH = TABLE_DIR / "12_volatility_yearly_results.csv"
SELECTIONS_PATH = TABLE_DIR / "12_volatility_selections.csv"
AUDIT_PATH = TABLE_DIR / "12_volatility_forecast_audit.csv"
COVERAGE_PATH = TABLE_DIR / "12_volatility_candidate_coverage.csv"

if not FINAL_PATH.exists():
    raise FileNotFoundError(
        f"Missing {FINAL_PATH}. Run Script 12 successfully first; do not use an older or manually edited predictions file."
    )

print("Project root:", ROOT)
print("Final predictions:", FINAL_PATH)
print("Development data:", DEV_PATH)
print("Holdout rule: Script 13 must not load any 2024–2025 observations.")

## 1. Load and validate Script 12 outputs
This cell checks the forecast schema, dates, duplicates, and positive forecast variances. It does not silently repair invalid rows.

In [ ]:
final = pd.read_csv(FINAL_PATH, parse_dates=["Forecast_Origin"])
dev = pd.read_csv(DEV_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

required = {
    "Currency", "Horizon", "Validation_Year", "Forecast_Origin",
    "Realized_Variance", "Blend_Forecast", "EWMA94_Forecast",
    "Forecast_Fallback_EWMA"
}
missing = required - set(final.columns)
if missing:
    raise ValueError(f"Final prediction file is missing required columns: {sorted(missing)}")

if dev["Date"].max() >= pd.Timestamp("2024-01-01"):
    raise RuntimeError(
        f"STOP: development dataset reaches {dev['Date'].max().date()}, "
        "which violates the locked 2024–2025 holdout boundary."
    )
if final["Forecast_Origin"].max() >= pd.Timestamp("2024-01-01"):
    raise RuntimeError(
        f"STOP: final predictions include {final['Forecast_Origin'].max().date()}; "
        "Script 13 is restricted to the 2019–2023 development evaluation."
    )

final["Validation_Year"] = pd.to_numeric(final["Validation_Year"], errors="coerce").astype("Int64")
final["Currency"] = final["Currency"].astype(str).str.upper()
final["Horizon"] = pd.to_numeric(final["Horizon"], errors="raise").astype(int)
final["Realized_Variance"] = pd.to_numeric(final["Realized_Variance"], errors="coerce")
final["Blend_Forecast"] = pd.to_numeric(final["Blend_Forecast"], errors="coerce")
final["EWMA94_Forecast"] = pd.to_numeric(final["EWMA94_Forecast"], errors="coerce")
final["Forecast_Fallback_EWMA"] = (
    final["Forecast_Fallback_EWMA"].astype(str).str.strip().str.lower().isin(["true", "1", "yes"])
)

key_cols = ["Currency", "Horizon", "Forecast_Origin"]
duplicate_count = int(final.duplicated(key_cols).sum())
if duplicate_count:
    raise ValueError(f"Found {duplicate_count} duplicate currency-horizon-origin rows in Script 12 final predictions.")

finite_target = np.isfinite(final["Realized_Variance"])
valid_variance = (
    np.isfinite(final["Blend_Forecast"]) & (final["Blend_Forecast"] > 0)
    & np.isfinite(final["EWMA94_Forecast"]) & (final["EWMA94_Forecast"] > 0)
)
valid_target_domain = final["Realized_Variance"] >= 0
final["Script13_Row_Valid"] = finite_target & valid_variance & valid_target_domain
invalid_rows = final.loc[~final["Script13_Row_Valid"]].copy()
valid = final.loc[final["Script13_Row_Valid"]].copy()

print("Development data:", dev["Date"].min().date(), "to", dev["Date"].max().date())
print("Final prediction rows:", len(final))
print("Valid rows used in metrics:", len(valid))
print("Invalid rows flagged (not silently discarded):", len(invalid_rows))
print("Forecast fallback rows:", int(valid["Forecast_Fallback_EWMA"].sum()))
print("Currencies:", sorted(valid["Currency"].unique().tolist()))
print("Horizons:", sorted(valid["Horizon"].unique().tolist()))

if invalid_rows.empty:
    print("PASS — all final prediction rows have valid targets and positive finite forecasts.")
else:
    display(invalid_rows.head(20))

## 2. Metric functions and primary pooled/yearly results
QLIKE is the primary volatility loss. Lower is better. Variance MSE and volatility RMSE are secondary metrics. The DM test is a diagnostic; its p-values should not be interpreted as proof of universal superiority.

In [ ]:
def qlike(rv, vf):
    rv = np.asarray(rv, dtype=float)
    vf = np.asarray(vf, dtype=float)
    m = np.isfinite(rv) & np.isfinite(vf) & (vf > 0) & (rv >= 0)
    if not m.any():
        return np.nan
    rv = np.maximum(rv[m], EPS)
    vf = np.maximum(vf[m], EPS)
    return float(np.mean(np.log(vf) + rv / vf))

def qlike_loss_vector(rv, vf):
    rv = np.maximum(np.asarray(rv, dtype=float), EPS)
    vf = np.maximum(np.asarray(vf, dtype=float), EPS)
    return np.log(vf) + rv / vf

def variance_mse(rv, vf):
    rv = np.asarray(rv, dtype=float)
    vf = np.asarray(vf, dtype=float)
    m = np.isfinite(rv) & np.isfinite(vf) & (rv >= 0) & (vf > 0)
    return float(np.mean((rv[m] - vf[m]) ** 2)) if m.any() else np.nan

def volatility_rmse(rv, vf):
    rv = np.asarray(rv, dtype=float)
    vf = np.asarray(vf, dtype=float)
    m = np.isfinite(rv) & np.isfinite(vf) & (rv >= 0) & (vf > 0)
    if not m.any():
        return np.nan
    return float(np.sqrt(np.mean((np.sqrt(np.maximum(rv[m], 0)) - np.sqrt(vf[m])) ** 2)))

def dm_qlike(rv, model_v, benchmark_v, horizon):
    rv = np.asarray(rv, dtype=float)
    model_v = np.asarray(model_v, dtype=float)
    benchmark_v = np.asarray(benchmark_v, dtype=float)
    m = np.isfinite(rv) & np.isfinite(model_v) & np.isfinite(benchmark_v) & (rv >= 0) & (model_v > 0) & (benchmark_v > 0)
    rv, model_v, benchmark_v = rv[m], model_v[m], benchmark_v[m]
    if len(rv) < 30:
        return np.nan, np.nan
    d = qlike_loss_vector(rv, benchmark_v) - qlike_loss_vector(rv, model_v)  # positive means final forecast is better
    n = len(d)
    lag = max(0, int(horizon) - 1)
    long_run_var = float(np.var(d, ddof=1))
    for k in range(1, min(lag, n - 1) + 1):
        cov = float(np.cov(d[k:], d[:-k], ddof=1)[0, 1])
        long_run_var += 2 * (1 - k / (lag + 1)) * cov
    if not np.isfinite(long_run_var) or long_run_var <= 0:
        return np.nan, np.nan
    stat = float(np.mean(d) / np.sqrt(long_run_var / n))
    pvalue = float(2 * stats.t.sf(abs(stat), df=n - 1))
    return stat, pvalue

def summarize_group(g, label_fields):
    rv = g["Realized_Variance"].to_numpy(float)
    fv = g["Blend_Forecast"].to_numpy(float)
    ev = g["EWMA94_Forecast"].to_numpy(float)
    stat, pvalue = dm_qlike(rv, fv, ev, int(g["Horizon"].iloc[0]))
    row = {field: g[field].iloc[0] for field in label_fields}
    row.update({
        "Observations": int(len(g)),
        "Fallback_Count": int(g["Forecast_Fallback_EWMA"].sum()),
        "Fallback_Rate": float(g["Forecast_Fallback_EWMA"].mean()),
        "EWMA_QLIKE": qlike(rv, ev),
        "Final_QLIKE": qlike(rv, fv),
        "QLIKE_Gain_vs_EWMA": qlike(rv, ev) - qlike(rv, fv),
        "EWMA_Variance_MSE": variance_mse(rv, ev),
        "Final_Variance_MSE": variance_mse(rv, fv),
        "MSE_Skill_vs_EWMA": 1 - variance_mse(rv, fv) / variance_mse(rv, ev) if variance_mse(rv, ev) > 0 else np.nan,
        "EWMA_Vol_RMSE": volatility_rmse(rv, ev),
        "Final_Vol_RMSE": volatility_rmse(rv, fv),
        "DM_Stat_QLIKE_vs_EWMA": stat,
        "DM_pvalue_QLIKE_vs_EWMA": pvalue,
        "Beats_EWMA_QLIKE": bool(qlike(rv, fv) < qlike(rv, ev)),
    })
    return row

pooled_rows = [
    summarize_group(g, ["Currency", "Horizon"])
    for _, g in valid.groupby(["Currency", "Horizon"], sort=True)
]
pooled = pd.DataFrame(pooled_rows).sort_values(["Currency", "Horizon"]).reset_index(drop=True)

yearly_rows = [
    summarize_group(g, ["Currency", "Horizon", "Validation_Year"])
    for _, g in valid.groupby(["Currency", "Horizon", "Validation_Year"], sort=True)
]
yearly = pd.DataFrame(yearly_rows).sort_values(["Currency", "Horizon", "Validation_Year"]).reset_index(drop=True)

display(pooled)
print("Pooled QLIKE wins vs EWMA:", int(pooled["Beats_EWMA_QLIKE"].sum()), "/", len(pooled))
display(yearly.head(20))

## 3. Reconciliation against Script 12's saved tables
This is a reproducibility check. If the current prediction file and saved pooled table are from the same run, the metric values should match within numerical tolerance. A mismatch is flagged for review; this script does not overwrite Script 12 outputs.

In [ ]:
reconciliation_rows = []
if POOLED_PATH.exists():
    old = pd.read_csv(POOLED_PATH)
    compare_cols = ["EWMA_QLIKE", "Final_QLIKE", "EWMA_Variance_MSE", "Final_Variance_MSE"]
    needed = {"Currency", "Horizon", *compare_cols}
    if needed.issubset(old.columns):
        joined = pooled.merge(old, on=["Currency", "Horizon"], suffixes=("_recomputed", "_saved"), how="outer", indicator=True)
        for _, row in joined.iterrows():
            diffs = {}
            for col in compare_cols:
                a, b = row.get(f"{col}_recomputed", np.nan), row.get(f"{col}_saved", np.nan)
                diffs[f"{col}_AbsDiff"] = abs(float(a) - float(b)) if pd.notna(a) and pd.notna(b) else np.nan
            reconciliation_rows.append({
                "Currency": row["Currency"], "Horizon": row["Horizon"],
                "Merge_Status": row["_merge"],
                **diffs,
                "Max_Abs_Difference": np.nanmax(list(diffs.values())) if any(pd.notna(v) for v in diffs.values()) else np.nan
            })
    else:
        print("Saved pooled table exists but has a different schema; skipping direct metric reconciliation.")
else:
    print("Saved Script 12 pooled table not found; the recomputed metrics will still be saved.")

reconciliation = pd.DataFrame(reconciliation_rows)
if not reconciliation.empty:
    display(reconciliation)
    max_diff = reconciliation["Max_Abs_Difference"].max()
    recon_status = "PASS" if reconciliation["Merge_Status"].eq("both").all() and pd.notna(max_diff) and max_diff < 1e-5 else "AUDIT"
    print("Reconciliation status:", recon_status, "| maximum absolute difference:", max_diff)
else:
    recon_status = "NOT_AVAILABLE"
    max_diff = np.nan

## 4. 2022 shock-period sensitivity
The primary pooled results retain every valid 2019–2023 forecast. The 2022 versus non-2022 split is a supplementary diagnostic only; it must not replace the primary evaluation or be used to tune the strategy.

In [ ]:
valid["Period_Group"] = np.where(valid["Validation_Year"].astype(int).eq(2022), "2022 shock year", "Other evaluation years")
period_rows = [
    summarize_group(g, ["Currency", "Horizon", "Period_Group"])
    for _, g in valid.groupby(["Currency", "Horizon", "Period_Group"], sort=True)
]
period_sensitivity = pd.DataFrame(period_rows).sort_values(["Currency", "Horizon", "Period_Group"]).reset_index(drop=True)
display(period_sensitivity)

# Supplementary pooled scores excluding 2022, to quantify how much the shock year changes the summary.
ex2022 = valid[valid["Validation_Year"].astype(int) != 2022]
ex2022_rows = [
    summarize_group(g, ["Currency", "Horizon"])
    for _, g in ex2022.groupby(["Currency", "Horizon"], sort=True)
]
excluding_2022 = pd.DataFrame(ex2022_rows).sort_values(["Currency", "Horizon"]).reset_index(drop=True)
excluding_2022 = excluding_2022.rename(columns={
    "Observations": "Observations_Excluding_2022",
    "EWMA_QLIKE": "EWMA_QLIKE_Excluding_2022",
    "Final_QLIKE": "Final_QLIKE_Excluding_2022",
    "QLIKE_Gain_vs_EWMA": "QLIKE_Gain_Excluding_2022",
    "MSE_Skill_vs_EWMA": "MSE_Skill_Excluding_2022",
    "Beats_EWMA_QLIKE": "Beats_EWMA_QLIKE_Excluding_2022"
})
display(excluding_2022[["Currency", "Horizon", "Observations_Excluding_2022", "EWMA_QLIKE_Excluding_2022",
                        "Final_QLIKE_Excluding_2022", "QLIKE_Gain_Excluding_2022",
                        "MSE_Skill_Excluding_2022", "Beats_EWMA_QLIKE_Excluding_2022"]])

## 5. Past-only normal/high volatility regime analysis
For each currency, compute the trailing 20-observation standard deviation of daily log returns, using returns available through the forecast origin. For each validation year, define the high-volatility threshold as the 75th percentile of that rolling volatility **strictly before that year's start**. This prevents future observations within a validation year from setting its regime threshold.

In [ ]:
regime_parts = []
for cur in CURRENCIES:
    mid_col = f"{cur}_mid"
    if mid_col not in dev.columns:
        raise KeyError(f"Development dataset is missing required column {mid_col}.")
    tmp = dev[["Date", mid_col]].copy()
    mid = pd.to_numeric(tmp[mid_col], errors="coerce")
    ret = 100 * np.log(mid / mid.shift(1))
    tmp["Currency"] = cur
    tmp["Return"] = ret
    tmp["Trailing20_ReturnVol"] = ret.rolling(20, min_periods=20).std()
    regime_parts.append(tmp[["Date", "Currency", "Return", "Trailing20_ReturnVol"]])

regime_daily = pd.concat(regime_parts, ignore_index=True)
regime_daily = regime_daily.rename(columns={"Date": "Forecast_Origin"})
regime_joined = valid.merge(
    regime_daily[["Currency", "Forecast_Origin", "Trailing20_ReturnVol"]],
    on=["Currency", "Forecast_Origin"], how="left", validate="many_to_one"
)

threshold_rows = []
for cur in CURRENCIES:
    for year in sorted(regime_joined.loc[regime_joined["Currency"].eq(cur), "Validation_Year"].dropna().astype(int).unique()):
        prior = regime_daily[
            (regime_daily["Currency"] == cur)
            & (regime_daily["Forecast_Origin"] < pd.Timestamp(year=year, month=1, day=1))
        ]["Trailing20_ReturnVol"].dropna()
        threshold = float(prior.quantile(HIGH_VOL_PERCENTILE)) if len(prior) >= 60 else np.nan
        threshold_rows.append({
            "Currency": cur, "Validation_Year": year,
            "Threshold_Training_Observations": int(len(prior)),
            "HighVol_Threshold_Trailing20Std": threshold,
            "Threshold_Percentile": HIGH_VOL_PERCENTILE,
            "Threshold_Uses_Prior_Years_Only": True
        })

thresholds = pd.DataFrame(threshold_rows)
regime_joined = regime_joined.merge(thresholds, on=["Currency", "Validation_Year"], how="left", validate="many_to_one")
regime_joined["Volatility_Regime"] = np.where(
    regime_joined["Trailing20_ReturnVol"].isna() | regime_joined["HighVol_Threshold_Trailing20Std"].isna(),
    "Unclassified",
    np.where(
        regime_joined["Trailing20_ReturnVol"] >= regime_joined["HighVol_Threshold_Trailing20Std"],
        "High volatility", "Normal volatility"
    )
)

regime_eval = regime_joined[regime_joined["Volatility_Regime"].isin(["High volatility", "Normal volatility"])].copy()
regime_rows = [
    summarize_group(g, ["Currency", "Horizon", "Volatility_Regime"])
    for _, g in regime_eval.groupby(["Currency", "Horizon", "Volatility_Regime"], sort=True)
]
regime_results = pd.DataFrame(regime_rows).sort_values(["Currency", "Horizon", "Volatility_Regime"]).reset_index(drop=True)
display(thresholds.head(12))
display(regime_results)
print("Unclassified regime rows:", int(regime_joined["Volatility_Regime"].eq("Unclassified").sum()))

## 6. Fallback and forecast-audit sensitivity
EWMA fallback is part of the declared operational strategy, so the primary result includes those fallback origins. The non-fallback-only result below is a secondary diagnostic and is conditional on successful component availability; it must not be presented as the primary unbiased comparison.

In [ ]:
fallback_rows = []
for (cur, h, year), g in valid.groupby(["Currency", "Horizon", "Validation_Year"], sort=True):
    fallback_rows.append({
        "Currency": cur, "Horizon": int(h), "Validation_Year": int(year),
        "Origins": int(len(g)),
        "Fallback_Count": int(g["Forecast_Fallback_EWMA"].sum()),
        "Fallback_Rate": float(g["Forecast_Fallback_EWMA"].mean()),
        "Primary_Includes_Fallbacks": True
    })
fallback_by_year = pd.DataFrame(fallback_rows).sort_values(["Currency", "Horizon", "Validation_Year"])
fallback_overall = (
    valid.groupby(["Currency", "Horizon"], as_index=False)
    .agg(Origins=("Forecast_Origin", "count"), Fallback_Count=("Forecast_Fallback_EWMA", "sum"))
)
fallback_overall["Fallback_Rate"] = fallback_overall["Fallback_Count"] / fallback_overall["Origins"]
display(fallback_overall)
display(fallback_by_year.head(20))

nonfallback = valid[~valid["Forecast_Fallback_EWMA"]].copy()
nonfallback_rows = [
    summarize_group(g, ["Currency", "Horizon"])
    for _, g in nonfallback.groupby(["Currency", "Horizon"], sort=True)
]
nonfallback_sensitivity = pd.DataFrame(nonfallback_rows).sort_values(["Currency", "Horizon"]).reset_index(drop=True)
if not nonfallback_sensitivity.empty:
    nonfallback_sensitivity = nonfallback_sensitivity.rename(columns={
        "Observations": "NonFallback_Observations",
        "Fallback_Count": "Fallback_Count_In_Subset",
        "Fallback_Rate": "Fallback_Rate_In_Subset",
        "EWMA_QLIKE": "EWMA_QLIKE_NonFallback_Origins",
        "Final_QLIKE": "Final_QLIKE_NonFallback_Origins",
        "QLIKE_Gain_vs_EWMA": "QLIKE_Gain_NonFallback_Origins",
        "MSE_Skill_vs_EWMA": "MSE_Skill_NonFallback_Origins",
        "Beats_EWMA_QLIKE": "Beats_EWMA_QLIKE_NonFallback_Origins"
    })
    display(nonfallback_sensitivity[["Currency", "Horizon", "NonFallback_Observations",
                                    "EWMA_QLIKE_NonFallback_Origins", "Final_QLIKE_NonFallback_Origins",
                                    "QLIKE_Gain_NonFallback_Origins", "MSE_Skill_NonFallback_Origins",
                                    "Beats_EWMA_QLIKE_NonFallback_Origins"]])

audit_summary = pd.DataFrame()
if AUDIT_PATH.exists():
    audit = pd.read_csv(AUDIT_PATH)
    if {"Model", "Stage", "Reason"}.issubset(audit.columns):
        audit_summary = (
            audit.groupby(["Model", "Stage", "Reason"], dropna=False)
            .size().reset_index(name="Audit_Entries")
            .sort_values("Audit_Entries", ascending=False)
        )
        display(audit_summary.head(30))
else:
    print("Forecast audit not found; skipped issue-count summary.")

selection_summary = pd.DataFrame()
if SELECTIONS_PATH.exists():
    selections = pd.read_csv(SELECTIONS_PATH)
    if {"Selected_Strategy", "Selected_Models"}.issubset(selections.columns):
        selection_summary = (
            selections.groupby(["Selected_Strategy", "Selected_Models"], dropna=False)
            .size().reset_index(name="Selection_Count")
            .sort_values("Selection_Count", ascending=False)
        )
        display(selection_summary.head(30))

## 7. QLIKE contribution and influence audit

This audit decomposes the primary QLIKE score into per-forecast-origin contributions for both the final volatility strategy and EWMA94. It identifies origins with unusually large losses, checks whether forecasts are extremely small relative to realized variance, and measures how much the largest contributions influence the pooled averages.

**Important:** this is a diagnostic only. The primary evaluation still includes every valid observation, including 2022 and operational EWMA fallbacks. The top-contribution-excluded scores are sensitivity diagnostics, not replacement results; no observation is deleted from the primary results and no model is tuned here.


In [ ]:
# Per-origin QLIKE contributions: same formula and EPS as the primary Script 12/13 metric.
contrib = valid.copy()
contrib["Realized_Variance_Used"] = np.maximum(
    pd.to_numeric(contrib["Realized_Variance"], errors="coerce").to_numpy(float), EPS
)
contrib["Final_Forecast_Used"] = np.maximum(
    pd.to_numeric(contrib["Blend_Forecast"], errors="coerce").to_numpy(float), EPS
)
contrib["EWMA_Forecast_Used"] = np.maximum(
    pd.to_numeric(contrib["EWMA94_Forecast"], errors="coerce").to_numpy(float), EPS
)

contrib["QLIKE_Final_Contribution"] = (
    np.log(contrib["Final_Forecast_Used"])
    + contrib["Realized_Variance_Used"] / contrib["Final_Forecast_Used"]
)
contrib["QLIKE_EWMA_Contribution"] = (
    np.log(contrib["EWMA_Forecast_Used"])
    + contrib["Realized_Variance_Used"] / contrib["EWMA_Forecast_Used"]
)
# Positive gain contribution means the final strategy has a lower loss than EWMA on this origin.
contrib["QLIKE_Gain_Contribution_vs_EWMA"] = (
    contrib["QLIKE_EWMA_Contribution"] - contrib["QLIKE_Final_Contribution"]
)
contrib["RV_to_Final_Forecast_Ratio"] = (
    contrib["Realized_Variance_Used"] / contrib["Final_Forecast_Used"]
)
contrib["RV_to_EWMA_Forecast_Ratio"] = (
    contrib["Realized_Variance_Used"] / contrib["EWMA_Forecast_Used"]
)
contrib["Final_to_EWMA_Forecast_Ratio"] = (
    contrib["Final_Forecast_Used"] / contrib["EWMA_Forecast_Used"]
)

# Rank each forecast origin within its currency-horizon task.
task_keys = ["Currency", "Horizon"]
contrib["Rank_Final_QLIKE_Descending"] = (
    contrib.groupby(task_keys)["QLIKE_Final_Contribution"]
    .rank(method="first", ascending=False).astype(int)
)
contrib["Rank_EWMA_QLIKE_Descending"] = (
    contrib.groupby(task_keys)["QLIKE_EWMA_Contribution"]
    .rank(method="first", ascending=False).astype(int)
)
contrib["Final_QLIKE_Above_Task_P99"] = False
contrib["EWMA_QLIKE_Above_Task_P99"] = False
for _, idx in contrib.groupby(task_keys).groups.items():
    final_p99 = contrib.loc[idx, "QLIKE_Final_Contribution"].quantile(0.99)
    ewma_p99 = contrib.loc[idx, "QLIKE_EWMA_Contribution"].quantile(0.99)
    contrib.loc[idx, "Final_QLIKE_Above_Task_P99"] = (
        contrib.loc[idx, "QLIKE_Final_Contribution"] >= final_p99
    )
    contrib.loc[idx, "EWMA_QLIKE_Above_Task_P99"] = (
        contrib.loc[idx, "QLIKE_EWMA_Contribution"] >= ewma_p99
    )

contribution_summary_rows = []
for (cur, horizon), g in contrib.groupby(task_keys, sort=True):
    g = g.sort_values("QLIKE_Final_Contribution", ascending=False).copy()
    n = len(g)
    positive_final = g.loc[g["QLIKE_Final_Contribution"] > 0, "QLIKE_Final_Contribution"]
    positive_total = float(positive_final.sum())
    row = {
        "Currency": cur,
        "Horizon": int(horizon),
        "Observations": int(n),
        "Final_QLIKE_Mean": float(g["QLIKE_Final_Contribution"].mean()),
        "Final_QLIKE_Median": float(g["QLIKE_Final_Contribution"].median()),
        "Final_QLIKE_P95": float(g["QLIKE_Final_Contribution"].quantile(0.95)),
        "Final_QLIKE_P99": float(g["QLIKE_Final_Contribution"].quantile(0.99)),
        "Final_QLIKE_Max": float(g["QLIKE_Final_Contribution"].max()),
        "EWMA_QLIKE_Mean": float(g["QLIKE_EWMA_Contribution"].mean()),
        "QLIKE_Gain_vs_EWMA": float(g["QLIKE_Gain_Contribution_vs_EWMA"].mean()),
        "Final_Forecast_Min": float(g["Final_Forecast_Used"].min()),
        "Final_Forecast_P01": float(g["Final_Forecast_Used"].quantile(0.01)),
        "Final_Forecast_Median": float(g["Final_Forecast_Used"].median()),
        "Final_Forecast_Max": float(g["Final_Forecast_Used"].max()),
        "EWMA_Forecast_Min": float(g["EWMA_Forecast_Used"].min()),
        "Realized_Variance_Max": float(g["Realized_Variance_Used"].max()),
        "Max_RV_to_Final_Forecast_Ratio": float(g["RV_to_Final_Forecast_Ratio"].max()),
        "Max_RV_to_EWMA_Forecast_Ratio": float(g["RV_to_EWMA_Forecast_Ratio"].max()),
        "Min_Final_to_EWMA_Forecast_Ratio": float(g["Final_to_EWMA_Forecast_Ratio"].min()),
        "Final_P99_Contribution_Count": int(g["Final_QLIKE_Above_Task_P99"].sum()),
        "EWMA_P99_Contribution_Count": int(g["EWMA_QLIKE_Above_Task_P99"].sum()),
        "Positive_Final_Loss_Total": positive_total,
        "Top1_PositiveLoss_Share": float(positive_final.nlargest(1).sum() / positive_total) if positive_total > 0 else np.nan,
        "Top5_PositiveLoss_Share": float(positive_final.nlargest(5).sum() / positive_total) if positive_total > 0 else np.nan,
        "Top10_PositiveLoss_Share": float(positive_final.nlargest(10).sum() / positive_total) if positive_total > 0 else np.nan,
    }

    # Influence diagnostics only: remove the largest final-strategy loss contributions
    # from BOTH strategies on the same origins to see whether the comparison is concentrated.
    for k in (1, 5, 10):
        kept = g.iloc[k:] if n > k else g.iloc[0:0]
        if not kept.empty:
            row[f"Final_QLIKE_Mean_Excl_Top{k}_FinalLoss_Origins"] = float(kept["QLIKE_Final_Contribution"].mean())
            row[f"EWMA_QLIKE_Mean_SameOrigins_Excl_Top{k}"] = float(kept["QLIKE_EWMA_Contribution"].mean())
            row[f"QLIKE_Gain_SameOrigins_Excl_Top{k}"] = float(
                kept["QLIKE_EWMA_Contribution"].mean() - kept["QLIKE_Final_Contribution"].mean()
            )
        else:
            row[f"Final_QLIKE_Mean_Excl_Top{k}_FinalLoss_Origins"] = np.nan
            row[f"EWMA_QLIKE_Mean_SameOrigins_Excl_Top{k}"] = np.nan
            row[f"QLIKE_Gain_SameOrigins_Excl_Top{k}"] = np.nan
    contribution_summary_rows.append(row)

qlike_contribution_summary = pd.DataFrame(contribution_summary_rows).sort_values(task_keys).reset_index(drop=True)

# Export the top 10 origins by each strategy's own QLIKE loss for every task.
top_final = contrib[contrib["Rank_Final_QLIKE_Descending"] <= 10].copy()
top_final["Top_Reason"] = "Top 10 final-strategy QLIKE contributions"
top_ewma = contrib[contrib["Rank_EWMA_QLIKE_Descending"] <= 10].copy()
top_ewma["Top_Reason"] = "Top 10 EWMA QLIKE contributions"
qlike_top_rows = pd.concat([top_final, top_ewma], ignore_index=True)
qlike_top_rows = qlike_top_rows.drop_duplicates(
    subset=["Currency", "Horizon", "Forecast_Origin", "Top_Reason"]
).sort_values(["Currency", "Horizon", "Top_Reason", "QLIKE_Final_Contribution"], ascending=[True, True, True, False])

# A concise review table for forecast origins with the largest final-strategy losses globally.
qlike_global_worst = contrib.sort_values("QLIKE_Final_Contribution", ascending=False).head(100).copy()

display(qlike_contribution_summary)
display(qlike_top_rows[[
    "Currency", "Horizon", "Validation_Year", "Forecast_Origin",
    "Realized_Variance", "Blend_Forecast", "EWMA94_Forecast",
    "QLIKE_Final_Contribution", "QLIKE_EWMA_Contribution",
    "QLIKE_Gain_Contribution_vs_EWMA", "RV_to_Final_Forecast_Ratio",
    "Final_to_EWMA_Forecast_Ratio", "Forecast_Fallback_EWMA", "Top_Reason"
]].head(40))
display(qlike_global_worst[[
    "Currency", "Horizon", "Validation_Year", "Forecast_Origin",
    "Realized_Variance", "Blend_Forecast", "EWMA94_Forecast",
    "QLIKE_Final_Contribution", "QLIKE_EWMA_Contribution",
    "QLIKE_Gain_Contribution_vs_EWMA", "RV_to_Final_Forecast_Ratio",
    "Final_to_EWMA_Forecast_Ratio", "Forecast_Fallback_EWMA"
]].head(30))

# Concentration chart: share of the total positive per-origin QLIKE contributions.
plot_audit = qlike_contribution_summary.copy()
plot_audit["Task"] = plot_audit["Currency"] + " H" + plot_audit["Horizon"].astype(str)
x = np.arange(len(plot_audit))
width = 0.24
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.bar(x - width, 100 * plot_audit["Top1_PositiveLoss_Share"], width, label="Top 1 origin")
ax.bar(x, 100 * plot_audit["Top5_PositiveLoss_Share"], width, label="Top 5 origins")
ax.bar(x + width, 100 * plot_audit["Top10_PositiveLoss_Share"], width, label="Top 10 origins")
ax.set_xticks(x)
ax.set_xticklabels(plot_audit["Task"], rotation=45, ha="right")
ax.set_ylabel("Share of positive final-strategy QLIKE contributions (%)")
ax.set_title("QLIKE Contribution Concentration Screen — Diagnostic Only")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "13_qlike_loss_concentration.png", dpi=180)
plt.show()
plt.close(fig)

# Save the full per-origin audit and summary. Primary Script 12/13 results are not overwritten.
contrib.to_csv(TABLE_DIR / "13_qlike_contributions_by_origin.csv", index=False)
qlike_top_rows.to_csv(TABLE_DIR / "13_qlike_top_contribution_origins_by_task.csv", index=False)
qlike_global_worst.to_csv(TABLE_DIR / "13_qlike_global_worst_100_origins.csv", index=False)
qlike_contribution_summary.to_csv(TABLE_DIR / "13_qlike_contribution_summary.csv", index=False)

# Screening thresholds are deliberately heuristic, not formal statistical decision rules.
concentration_flags = qlike_contribution_summary[
    (qlike_contribution_summary["Top1_PositiveLoss_Share"] >= 0.10)
    | (qlike_contribution_summary["Top10_PositiveLoss_Share"] >= 0.50)
][["Currency", "Horizon", "Top1_PositiveLoss_Share", "Top5_PositiveLoss_Share", "Top10_PositiveLoss_Share"]].copy()

print("QLIKE contribution audit saved.")
print("Tasks triggering the descriptive concentration screen:", len(concentration_flags), "of", len(qlike_contribution_summary))
if not concentration_flags.empty:
    display(concentration_flags)
print("Reminder: top-contribution-excluded values are diagnostics only; all valid origins remain in primary results.")


## 8. Figures and saved outputs
The primary pooled result includes 2022 and all operational EWMA fallback origins. The supplementary analyses are not used to select or retune the model.
The QLIKE contribution audit additionally saves `13_qlike_contributions_by_origin.csv`, `13_qlike_top_contribution_origins_by_task.csv`, `13_qlike_global_worst_100_origins.csv`, `13_qlike_contribution_summary.csv`, and `13_qlike_loss_concentration.png`.


In [ ]:
# Figure 1: pooled QLIKE gain by currency and horizon.
plot_df = pooled.copy()
plot_df["Task"] = plot_df["Currency"] + " H" + plot_df["Horizon"].astype(str)
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(plot_df["Task"], plot_df["QLIKE_Gain_vs_EWMA"])
ax.axhline(0, linewidth=1)
ax.set_title("Script 12 Strategy vs EWMA — Pooled QLIKE Gain (2019–2023)")
ax.set_ylabel("EWMA QLIKE − Final QLIKE (positive favours final strategy)")
ax.set_xlabel("Currency–horizon task")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIG_DIR / "13_volatility_pooled_qlike_gain.png", dpi=180)
plt.show()
plt.close(fig)

# Figure 2: yearly QLIKE gains, preserving all years including 2022.
yearly_plot = yearly.copy()
yearly_plot["Task"] = yearly_plot["Currency"] + " H" + yearly_plot["Horizon"].astype(str)
fig, ax = plt.subplots(figsize=(12, 6))
for task, group in yearly_plot.groupby("Task"):
    ax.plot(group["Validation_Year"].astype(int), group["QLIKE_Gain_vs_EWMA"], marker="o", label=task)
ax.axhline(0, linewidth=1)
ax.set_title("Yearly QLIKE Gain vs EWMA (2022 retained)")
ax.set_xlabel("Validation year")
ax.set_ylabel("EWMA QLIKE − Final QLIKE")
ax.legend(ncol=4, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "13_volatility_yearly_qlike_stability.png", dpi=180)
plt.show()
plt.close(fig)

# Save new Script 13 files only.
pooled.to_csv(TABLE_DIR / "13_volatility_sensitivity_pooled.csv", index=False)
yearly.to_csv(TABLE_DIR / "13_volatility_sensitivity_yearly.csv", index=False)
period_sensitivity.to_csv(TABLE_DIR / "13_volatility_2022_vs_other_years.csv", index=False)
excluding_2022.to_csv(TABLE_DIR / "13_volatility_excluding_2022_diagnostic.csv", index=False)
thresholds.to_csv(TABLE_DIR / "13_volatility_regime_thresholds.csv", index=False)
regime_results.to_csv(TABLE_DIR / "13_volatility_regime_results.csv", index=False)
fallback_by_year.to_csv(TABLE_DIR / "13_volatility_fallback_by_year.csv", index=False)
fallback_overall.to_csv(TABLE_DIR / "13_volatility_fallback_summary.csv", index=False)
nonfallback_sensitivity.to_csv(TABLE_DIR / "13_volatility_nonfallback_sensitivity.csv", index=False)
if not reconciliation.empty:
    reconciliation.to_csv(TABLE_DIR / "13_volatility_pooled_reconciliation.csv", index=False)
if not audit_summary.empty:
    audit_summary.to_csv(TABLE_DIR / "13_volatility_audit_summary.csv", index=False)
if not selection_summary.empty:
    selection_summary.to_csv(TABLE_DIR / "13_volatility_selection_summary.csv", index=False)
if not invalid_rows.empty:
    invalid_rows.to_csv(TABLE_DIR / "13_volatility_invalid_prediction_rows.csv", index=False)

print("Script 13 output tables saved to:", TABLE_DIR)
print("Script 13 figures saved to:", FIG_DIR)

## 9. Final quality gate
This gate is deliberately descriptive. Audit warnings in Script 12 do not disappear because the sensitivity notebook runs. If pooled results do not reproduce, investigate the same-run outputs before freezing the methodology.

In [ ]:
quality_rows = []
max_date = valid["Forecast_Origin"].max() if not valid.empty else pd.NaT
quality_rows.append({
    "Check": "Holdout isolation",
    "Status": "PASS" if pd.notna(max_date) and max_date < pd.Timestamp("2024-01-01") and dev["Date"].max() < pd.Timestamp("2024-01-01") else "FAIL",
    "Evidence": f"Latest valid forecast origin={max_date}; development max date={dev['Date'].max()}"
})
quality_rows.append({
    "Check": "All final forecasts valid",
    "Status": "PASS" if invalid_rows.empty else "AUDIT",
    "Evidence": f"Invalid rows={len(invalid_rows)} of {len(final)}"
})
task_counts = valid.groupby(["Currency", "Horizon"]).size()
quality_rows.append({
    "Check": "12 currency-horizon tasks present",
    "Status": "PASS" if len(task_counts) == 12 and all(task_counts.get((c, h), 0) > 0 for c in CURRENCIES for h in HORIZONS) else "AUDIT",
    "Evidence": f"Tasks present={len(task_counts)}; expected=12"
})
quality_rows.append({
    "Check": "Saved pooled table reproduces",
    "Status": recon_status,
    "Evidence": f"Maximum absolute difference across checked metrics={max_diff}"
})
quality_rows.append({
    "Check": "Fallbacks reported",
    "Status": "PASS" if "Forecast_Fallback_EWMA" in valid.columns else "FAIL",
    "Evidence": f"Fallback rows={int(valid['Forecast_Fallback_EWMA'].sum())}; rate={valid['Forecast_Fallback_EWMA'].mean():.4f}"
})
quality_rows.append({
    "Check": "2022 retained in primary evaluation",
    "Status": "PASS" if valid["Validation_Year"].astype(int).eq(2022).any() else "FAIL",
    "Evidence": "Primary pooled and yearly results include 2022; exclusion is supplementary only"
})
quality_rows.append({
    "Check": "Past-only volatility regime threshold",
    "Status": "PASS" if thresholds["Threshold_Uses_Prior_Years_Only"].all() else "FAIL",
    "Evidence": "Each validation-year threshold uses only observations before that year's start"
})
quality_rows.append({
    "Check": "Model instability audit",
    "Status": "AUDIT",
    "Evidence": "Script 12 fit/forecast rejection logs are summarized separately; this sensitivity script does not erase or override them"
})

concentration_status = "AUDIT" if not concentration_flags.empty else "PASS"
quality_rows.append({
    "Check": "QLIKE contribution concentration screen",
    "Status": concentration_status,
    "Evidence": (
        f"{len(concentration_flags)} of {len(qlike_contribution_summary)} currency-horizon tasks "
        "crossed the descriptive screen (Top 1 share >= 10% or Top 10 share >= 50%). "
        "These are heuristic review flags, not formal rejection thresholds; primary results retain all origins."
    )
})

quality_gate_13 = pd.DataFrame(quality_rows)
display(quality_gate_13)
quality_gate_13.to_csv(TABLE_DIR / "13_volatility_sensitivity_quality_gate.csv", index=False)

print("\\nSCRIPT 13 COMPLETE — SENSITIVITY AND REGIME-STABILITY ANALYSIS")
print("Primary pooled QLIKE wins:", int(pooled["Beats_EWMA_QLIKE"].sum()), "/", len(pooled))
print("2022 remains in primary evaluation.")
print("QLIKE contribution audit saved; inspect 13_qlike_contribution_summary.csv and 13_qlike_top_contribution_origins_by_task.csv.")
print("No model refitting or hyperparameter selection was performed.")
print("2024–2025 holdout was not loaded.")
print("Review 13_volatility_sensitivity_quality_gate.csv before proceeding to Script 14.")